Installations de package

Imports modèle + utilitaires

In [4]:
REPO_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM"
DATA_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/IMDB"

print("DATA_DIR:", DATA_DIR)
print("REPO_DIR:", REPO_DIR)


DATA_DIR: /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/IMDB
REPO_DIR: /Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM


In [6]:
import os
import numpy as np
import scipy.sparse as sp

def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_bow(path):
    X = sp.load_npz(path).astype(np.float32)  
    return X.toarray()                        


def load_word_embeddings(path):
    W = sp.load_npz(path)                 
    return W.astype(np.float32).toarray() 


train_bow = load_bow(os.path.join(DATA_DIR, "train_bow.npz"))
test_bow  = load_bow(os.path.join(DATA_DIR, "test_bow.npz"))
vocab     = load_vocab(os.path.join(DATA_DIR, "vocab.txt"))

print("train_bow:", train_bow.shape)
print("test_bow :", test_bow.shape)
print("vocab    :", len(vocab))


train_bow: (25000, 5000)
test_bow : (25000, 5000)
vocab    : 5000


In [7]:
!pip install torch torchvision torchaudio


In [8]:
import os, random
import numpy as np
import torch

REPO_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM" 
DATA_ROOT = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data"
OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs/ECTRM"
os.makedirs(OUT_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  
    return seed

SEED = set_seed(42)
print("SEED =", SEED)

SEED = 42


In [9]:
CONFIGS = {
  "20NG": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_20NG.yaml",
  "AGNews": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_AGNews.yaml",
  "IMBD": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_IMDB.yaml",
  "YahooAnswer": "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/ECRTM/configs/model/ECRTM_YahooAnswer.yaml",
}


In [10]:
import sys
import yaml
import scipy.io
import scipy.sparse as sp
from types import SimpleNamespace
from torch.utils.data import DataLoader, TensorDataset

sys.path.append(REPO_DIR)
from models.ECRTM import ECRTM

def load_cfg(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)  

def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_bow_npz(path):
    X = sp.load_npz(path).astype(np.float32)  
    return X.toarray()                       

def load_word_emb_npz(path):
    W = sp.load_npz(path)                 
    return W.astype(np.float32).toarray()

Fonctions: build args + infer theta

In [15]:
def build_args_from_yaml(cfg, K, vocab_size, word_emb):
    cfg = dict(cfg)  # copie
    cfg["num_topic"] = K
    cfg["vocab_size"] = vocab_size
    cfg["word_embeddings"] = word_emb
    return SimpleNamespace(**cfg)


def infer_theta(model, bow_np, device, batch_size=256):
    model.eval()
    thetas = []

    with torch.no_grad():
        # 👉 conversion sparse → dense UNE FOIS (minimal change)
        if hasattr(bow_np, "toarray"):
            bow_np = bow_np.toarray()

        X = torch.from_numpy(bow_np).float()
        loader = DataLoader(
            TensorDataset(X),
            batch_size=batch_size,
            shuffle=False
        )

        for (bow,) in loader:
            bow = bow.to(device)
            theta = model.get_theta(bow)
            thetas.append(theta.cpu().numpy())

    return np.vstack(thetas)

 run complet: train_one(dataset, K)

In [16]:
import os
import torch
import scipy.io

from rich.progress import (
    Progress,
    BarColumn,
    TimeElapsedColumn,
    TimeRemainingColumn,
    SpinnerColumn,
)
from rich.console import Console

console = Console()


def train_one(dataset, K, seed=42):
    set_seed(seed)

    # --- Dataset ---
    data_dir = DATASETS[dataset]
    cfg = load_cfg(CONFIGS[dataset])

    # --- Hyperparameters ---
    epochs = int(cfg["epochs"])
    batch_size = int(cfg["batch_size"])
    lr = float(cfg["learning_rate"])

    # --- Data loading ---
    train_bow = load_bow_npz(os.path.join(data_dir, "train_bow.npz"))
    test_bow  = load_bow_npz(os.path.join(data_dir, "test_bow.npz"))
    vocab     = load_vocab(os.path.join(data_dir, "vocab.txt"))
    word_emb  = load_word_emb_npz(os.path.join(data_dir, "word_embeddings.npz"))

    V = train_bow.shape[1]

    # --- Reshape embeddings ---
    emb_dim = word_emb.size // V
    word_emb = word_emb[: V * emb_dim].reshape(V, emb_dim)

    assert V == test_bow.shape[1] == len(vocab) == word_emb.shape[0], \
        "Mismatch vocab sizes"

    # --- Model ---
    args = build_args_from_yaml(cfg, K=K, vocab_size=V, word_emb=word_emb)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ECRTM(args).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # --- DataLoader ---
    X = torch.from_numpy(train_bow.toarray()).float()
    loader = DataLoader(
        TensorDataset(X),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device == "cuda"),
    )

    # --- Training (rich progress) ---
    with Progress(
        SpinnerColumn(),
        "[bold blue]{task.description}[/bold blue]",
        BarColumn(),
        "[progress.percentage]{task.percentage:>3.0f}%",
        TimeElapsedColumn(),
        TimeRemainingColumn(),
        console=console,
        refresh_per_second=5,
    ) as progress:

        epoch_task = progress.add_task(
            f"{dataset} | K={K} | seed={seed}",
            total=epochs,
        )

        for epoch in range(1, epochs + 1):
            model.train()
            total_loss = 0.0

            batch_task = progress.add_task(
                f"Epoch {epoch}/{epochs}",
                total=len(loader),
            )

            for (bow,) in loader:
                bow = bow.to(device, non_blocking=True)

                rst = model(bow)
                loss = rst["loss"]

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += float(loss.detach().cpu())
                progress.advance(batch_task, 1)

            progress.remove_task(batch_task)

            avg_loss = total_loss / len(loader)
            progress.advance(epoch_task, 1)
            progress.update(
                epoch_task,
                description=f"{dataset} | K={K} | loss={avg_loss:.1f}",
            )

    # --- Save results ---
    model.eval()
    with torch.no_grad():
        beta = model.get_beta().cpu().numpy()

    train_theta = infer_theta(
        model, train_bow, device=device, batch_size=256
    )
    test_theta = infer_theta(
        model, test_bow, device=device, batch_size=256
    )

    out_path = os.path.join(
        OUT_DIR, f"{dataset}_ECRTM_K{K}_seed{seed}.mat"
    )

    scipy.io.savemat(
        out_path,
        {
            "beta": beta,
            "train_theta": train_theta,
            "test_theta": test_theta,
        },
    )

    console.print(f"[green]Saved:[/green] {out_path}")
    return out_path


20NG

In [14]:
# for K in [20, 50, 100]:
   #  train_one("20NG", K=K, seed=42)

In [ ]:
beta_20   = results[20]["beta"]
theta_20  = results[20]["train_theta"]

beta_50   = results[50]["beta"]
theta_50  = results[50]["train_theta"]

beta_100  = results[100]["beta"]
theta_100 = results[100]["train_theta"]

In [ ]:
for K in results:
    print(
        f"K={K}",
        results[K]["beta"].shape,
        results[K]["train_theta"].shape
    )

Avec 11 314 documents et un vocabulaire de 5 000 mots, ECRTM apprend pour K=20 des thèmes larges et partagés, pour K=50 des thèmes plus discriminants, et pour K=100 des thèmes très spécialisés dont certains deviennent peu utilisés ou redondants, comme le suggère l’augmentation linéaire de β (K×5000) et de θ (11314×K).

IMDB

In [17]:
for K in [20, 50, 100]:
    train_one("IMDB", K=K, seed=42)

Output()

Saved: /kaggle/working/ECTRM_runs/IMDB_ECRTM_K20_seed42.mat

Output()

Saved: /kaggle/working/ECTRM_runs/IMDB_ECRTM_K50_seed42.mat

Output()

Saved: /kaggle/working/ECTRM_runs/IMDB_ECRTM_K100_seed42.mat

In [24]:
import scipy.io

# Paths vers les résultats IMDB
paths = {
    20: "/kaggle/working/ECTRM_runs/IMDB_ECRTM_K20_seed42.mat",
    50: "/kaggle/working/ECTRM_runs/IMDB_ECRTM_K50_seed42.mat",
    100: "/kaggle/working/ECTRM_runs/IMDB_ECRTM_K100_seed42.mat",
}

# Chargement des résultats
results_imdb = {}
for K, p in paths.items():
    results_imdb[K] = scipy.io.loadmat(p)

# Vérification des clés disponibles
print("Keys in K=20 file:", results_imdb[20].keys())

# Extraction beta / theta par K
beta_20   = results_imdb[20]["beta"]
theta_20  = results_imdb[20]["train_theta"]

beta_50   = results_imdb[50]["beta"]
theta_50  = results_imdb[50]["train_theta"]

beta_100  = results_imdb[100]["beta"]
theta_100 = results_imdb[100]["train_theta"]

# Sanity check des dimensions
for K in results_imdb:
    print(
        f"K={K}",
        results_imdb[K]["beta"].shape,
        results_imdb[K]["train_theta"].shape
    )


Keys in K=20 file: dict_keys(['__header__', '__version__', '__globals__', 'beta', 'train_theta', 'test_theta'])
K=20 (20, 5000) (25000, 20)
K=50 (50, 5000) (25000, 50)
K=100 (100, 5000) (25000, 100)


Sur le corpus IMDB, les résultats montrent que pour K=20 le modèle ECRTM capture des thèmes larges et transversaux partagés par de nombreuses critiques, tandis que pour K=50 il segmente plus finement les opinions, genres et registres lexicaux, et pour K=100 il apprend des micro-thèmes très spécialisés, parfois peu représentés, ce qui se reflète directement dans l’augmentation structurée de la matrice des topics β (de 20×5000 à 100×5000) et de la représentation document-topic θ (de 25 000×20 à 25 000×100).

 Imports + dossier de sortie LDA

In [ ]:
import os
import numpy as np
import scipy.io
import scipy.sparse as sp
from sklearn.decomposition import LatentDirichletAllocation

RUNS_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"  
os.makedirs(RUNS_DIR, exist_ok=True)
print("RUNS_DIR =", RUNS_DIR)

OUT_DIR_LDA = os.path.join(RUNS_DIR, "LDA")
os.makedirs(OUT_DIR_LDA, exist_ok=True)
print("OUT_DIR_LDA =", OUT_DIR_LDA)

Loader sparse (spécifique LDA)

In [ ]:
def load_bow_sparse(path):
    return sp.load_npz(path)


Fonction train_one_lda(dataset, K)

In [ ]:
def train_one_lda(dataset, K, seed=42, max_iter=50):
    data_dir = os.path.join(DATA_ROOT, dataset)

    X_train = load_bow_sparse(os.path.join(data_dir, "train_bow.npz"))
    X_test  = load_bow_sparse(os.path.join(data_dir, "test_bow.npz"))

    out_path = os.path.join(OUT_DIR_LDA, f"{dataset}_LDA_K{K}_seed{seed}.mat")
    if os.path.exists(out_path):
        print("Skip (already exists):", out_path)
        return out_path

    lda = LatentDirichletAllocation(
        n_components=K,           
        max_iter=max_iter,        
        learning_method="batch",
        random_state=seed,
        verbose=1                 
    )
    lda.fit(X_train)  

    beta = lda.components_.astype(np.float32)           
    beta = beta / beta.sum(axis=1, keepdims=True)

    train_theta = lda.transform(X_train).astype(np.float32)  
    test_theta  = lda.transform(X_test).astype(np.float32)   

    scipy.io.savemat(out_path, {"beta": beta, "train_theta": train_theta, "test_theta": test_theta})
    print("Saved:", out_path)
    return out_path


In [ ]:
def train_one_lda(dataset, K, seed=42, max_iter=50):
    data_dir = os.path.join(DATA_ROOT, dataset)

    X_train = load_bow_sparse(os.path.join(data_dir, "train_bow.npz"))
    X_test  = load_bow_sparse(os.path.join(data_dir, "test_bow.npz"))

    out_path = os.path.join(OUT_DIR_LDA, f"{dataset}_LDA_K{K}_seed{seed}.mat")
    if os.path.exists(out_path):
        print("Skip (already exists):", out_path)
        return out_path

    lda = LatentDirichletAllocation(
        n_components=K,           
        max_iter=max_iter,        
        learning_method="batch",
        random_state=seed,
        verbose=1                 
    )
    lda.fit(X_train) 

    beta = lda.components_.astype(np.float32)          
    beta = beta / beta.sum(axis=1, keepdims=True)

    train_theta = lda.transform(X_train).astype(np.float32)  
    test_theta  = lda.transform(X_test).astype(np.float32)   

    scipy.io.savemat(out_path, {"beta": beta, "train_theta": train_theta, "test_theta": test_theta})
    print("Saved:", out_path)
    return out_path


In [ ]:
for K in [20, 50, 100]:
    train_one_lda("YahooAnswer", K=K, seed=42, max_iter=50)


In [ ]:
import os, numpy as np
import scipy.io 

PROJECT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM"
VOCAB_PATH  = os.path.join(PROJECT_DIR, "data/20NG/vocab.txt")
TOPICS_FILE = os.path.join(PROJECT_DIR, "metrics/topics_LDA_for_cv.txt")
os.makedirs(os.path.dirname(TOPICS_FILE), exist_ok=True)

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f if line.strip()]


lda_output_path = os.path.join(OUT_DIR_LDA, "20NG_LDA_K20_seed42.mat")
loaded_data = scipy.io.loadmat(lda_output_path)
beta = loaded_data["beta"]  

TOPN = 10

with open(TOPICS_FILE, "w", encoding="utf-8") as f:
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:TOPN]
        f.write(" ".join(vocab[i] for i in top_idx) + "\n")

print("OK topics file:", TOPICS_FILE)

PALMETTO

In [ ]:
WIKI_DIR="/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/Wikipedia_bd"
!mkdir -p "$WIKI_DIR"

!ls -l "$WIKI_DIR/wikipedia_bd"

In [ ]:
!test -f "$PALMETTO_JAR" && echo "OK: jar" || echo "KO: jar"
!test -d "$WIKI_BD_DIR" && echo "OK: wikipedia_bd dir" || echo "KO: wikipedia_bd dir"
!test -f "$TOPICS_FILE" && echo "OK: topics file" || echo "KO: topics file"

!ls -lah "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/Wikipedia_bd" | head -n 50


In [ ]:
import os
import numpy as np
import scipy.io
import subprocess

#  1. Chemins 
PROJECT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM"
VOCAB_PATH  = os.path.join(PROJECT_DIR, "data/20NG/vocab.txt")

OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"
OUT_DIR_LDA   = f"{OUT_DIR}/LDA"
OUT_DIR_ECRTM = f"{OUT_DIR}/ECRTM"


METRICS_DIR = OUT_DIR  

os.makedirs(OUT_DIR_LDA, exist_ok=True)
os.makedirs(OUT_DIR_ECRTM, exist_ok=True)

#  2. Palmetto 
PALMETTO_JAR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/palmetto-0.1.0-jar-with-dependencies.jar"
WIKI_BD_DIR  = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/palmetto/Wikipedia_bd/wikipedia_bd"
TOPN = 15

#  3. Vocab 
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f if line.strip()]

#  4. Fichiers .mat 
lda_files = [
    "20NG_LDA_K20_seed42.mat",
    "20NG_LDA_K50_seed42.mat",
    "20NG_LDA_K100_seed42.mat",
]

ecrtm_files = [
    "20NG_ECRTM_K20_seed42.mat",
    "20NG_ECRTM_K50_seed42.mat",
    "20NG_ECRTM_K100_seed42.mat",
]

#  5. Fonction de processing 
def process_mat_files(mat_files, base_dir, model_name):
    for mat_name in mat_files:
        mat_path = os.path.join(base_dir, mat_name)
        
        if not os.path.isfile(mat_path):
            print(f"Introuvable: {mat_path}")
            continue
        
        loaded = scipy.io.loadmat(mat_path)
        if "beta" not in loaded:
            print(f"'beta' introuvable dans {mat_name}. Keys: {list(loaded.keys())}")
            continue
        
        beta = loaded["beta"] 
        
        base_name = mat_name.replace(".mat", "")
        
        # Sauvegarde dans le dossier (LDA ou ECRTM)
        topics_file = os.path.join(base_dir, f"topics_for_cv_{base_name}.txt")
        cv_out_file = os.path.join(base_dir, f"palmetto_CV_{base_name}.txt")
        
        # Écrire topics 
        with open(topics_file, "w", encoding="utf-8") as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(" ".join(vocab[i] for i in top_idx) + "\n") 
        
        # Lancer Palmetto
        cmd = ["java", "-jar", PALMETTO_JAR, WIKI_BD_DIR, "C_V", topics_file]
        try:
            out = subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)
            with open(cv_out_file, "w", encoding="utf-8") as f:
                f.write(out)
            print(f"{model_name} OK: {mat_name} -> {cv_out_file}")
        except subprocess.CalledProcessError as e:
            print(f"Erreur Palmetto pour {mat_name}:\n{e.output}")

#  6. Lancer 
print("=== Processing LDA ===")
process_mat_files(lda_files, OUT_DIR_LDA, "LDA")

print("\n=== Processing ECRTM ===")
process_mat_files(ecrtm_files, OUT_DIR_ECRTM, "ECRTM")

print("\nTerminé. Résultats dans:")
print(f"  LDA   → {OUT_DIR_LDA}")
print(f"  ECRTM → {OUT_DIR_ECRTM}")


In [ ]:
import os
import numpy as np
import pandas as pd

OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"
OUT_DIR_LDA   = f"{OUT_DIR}/LDA"
OUT_DIR_ECRTM = f"{OUT_DIR}/ECRTM"

configs = [
    ("LDA",   20, 42, OUT_DIR_LDA),
    ("LDA",   50, 42, OUT_DIR_LDA),
    ("LDA",  100, 42, OUT_DIR_LDA),
    ("ECRTM", 20, 42, OUT_DIR_ECRTM),
    ("ECRTM", 50, 42, OUT_DIR_ECRTM),
    ("ECRTM",100, 42, OUT_DIR_ECRTM),
]

rows = []

for model, K, seed, folder in configs:
    fn = f"palmetto_CV_20NG_{model}_K{K}_seed{seed}.txt"
    path = os.path.join(folder, fn)
    
    if not os.path.isfile(path):
        print(f" MANQUANT: {path}")
        continue
    
    print(f" Lecture: {fn}")
    
    vals = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            # Skip vides ou logs
            if not line or "INFO" in line or "ERROR" in line or line.startswith("2026-"):
                continue
            
            # Format Palmetto : "    0\t0,30745\t[mots...]"
            # On split sur TAB
            parts = line.split('\t')
            
            if len(parts) >= 2:
                # parts[0] = index, parts[1] = score
                score_str = parts[1].replace(',', '.')  # virgule → point
                try:
                    score = float(score_str)
                    vals.append(score)
                except ValueError:
                    pass
    
    if not vals:
        print(f"Aucun score trouvé dans {fn}")
        continue
    
    vals = np.array(vals, dtype=float)
    
    rows.append({
        "model": model,
        "K": K,
        "seed": seed,
        "CV_mean": float(vals.mean()),
        "CV_std": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
        "CV_min": float(vals.min()),
        "CV_max": float(vals.max()),
        "n_topics": len(vals),
    })

if not rows:
    print("\n Aucun score C_V trouvé.")
else:
    df_cv = pd.DataFrame(rows).sort_values(["model", "K"])
    out_csv = os.path.join(OUT_DIR, "CV_summary.csv")
    df_cv.to_csv(out_csv, index=False)
    
    print(f"\n Résumé sauvegardé: {out_csv}\n")
    print(df_cv.to_string(index=False))


In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.io
from sklearn.metrics import normalized_mutual_info_score

#  PATHS 
PROJECT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project"
OUT_DIR = "/Users/hydersidra/Documents/Master/PPD/ECRTM_runs"

OUT_DIR_LDA   = f"{OUT_DIR}/LDA"
OUT_DIR_ECRTM = f"{OUT_DIR}/ECRTM"

VOCAB_PATH  = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/20NG/vocab.txt"
LABELS_PATH = "/Users/hydersidra/Documents/Master/PPD/ECRTM_project/repo/ECRTM/data/20NG/test_labels.txt"

TOPN_TD = 25  # Topic Diversity sur top-25 mots (standard papiers)

#  LOAD DATA 
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = [line.strip() for line in f if line.strip()]

y_true = np.loadtxt(LABELS_PATH, dtype=int)
print(f"Chargé: {len(vocab)} mots vocab, {len(y_true)} documents avec labels\n")

#  METRICS 
def topic_diversity_from_beta(beta, vocab, topn=25):
    """TD = proportion de mots uniques dans les top-N de tous les topics"""
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(vocab[i] for i in top_idx)
    return len(set(all_words)) / max(1, len(all_words))

def purity_score(y_true, y_pred):
    """Purity = proportion de docs dans le cluster majoritaire pour chaque cluster"""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    purity = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        if len(idx) == 0:
            continue
        _, counts = np.unique(y_true[idx], return_counts=True)
        purity += counts.max()
    return purity / len(y_true)

def find_theta_in_mat(d, K, N):
    """Cherche la matrice doc-topic (N, K) dans le .mat"""
    skip = {"__header__", "__version__", "__globals__"}
    for name, arr in d.items():
        if name in skip or not isinstance(arr, np.ndarray) or arr.ndim != 2:
            continue
        if arr.shape == (N, K):
            return name, arr
        if arr.shape == (K, N):
            return name, arr.T
    raise ValueError(f"Theta (doc-topic) introuvable. Clés disponibles: {list(d.keys())}")

#  CONFIG 
configs = [
    ("LDA",   20, 42, OUT_DIR_LDA),
    ("LDA",   50, 42, OUT_DIR_LDA),
    ("LDA",  100, 42, OUT_DIR_LDA),
    ("ECRTM", 20, 42, OUT_DIR_ECRTM),
    ("ECRTM", 50, 42, OUT_DIR_ECRTM),
    ("ECRTM",100, 42, OUT_DIR_ECRTM),
]

rows = []

for model, K, seed, folder in configs:
    fn = f"20NG_{model}_K{K}_seed{seed}.mat"
    mat_path = os.path.join(folder, fn)
    
    if not os.path.isfile(mat_path):
        print(f" MANQUANT: {mat_path}")
        continue
    
    print(f" Lecture: {fn}")
    
    d = scipy.io.loadmat(mat_path)
    beta = d["beta"]  # (K, V)
    
    # Trouver theta (doc-topic)
    try:
        theta_key, theta = find_theta_in_mat(d, K, len(y_true))
    except ValueError as e:
        print(f" Erreur pour {fn}: {e}")
        continue
    
    # Clustering: chaque doc assigné au topic dominant
    y_pred = theta.argmax(axis=1)
    
    # Métriques
    td = topic_diversity_from_beta(beta, vocab, topn=TOPN_TD)
    purity = purity_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    
    rows.append({
        "model": model,
        "K": K,
        "seed": seed,
        "TD": round(td, 4),
        "Purity": round(purity, 4),
        "NMI": round(nmi, 4),
    })

#  SAVE & DISPLAY
if not rows:
    print("\n Aucune métrique calculée.")
else:
    df = pd.DataFrame(rows).sort_values(["model", "K"])
    
    out_csv = os.path.join(PROJECT_DIR, "metrics", "TD_Purity_NMI.csv")
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    df.to_csv(out_csv, index=False)
    
    print(f"\n Résumé sauvegardé: {out_csv}\n")
    print(df.to_string(index=False))
